# RNA-Hi-C Conditional Diffusion Pipeline & Faz 1 vs Faz 2 Adil Kıyaslama (`rna-diffusion-clean`)

Bu notebook **Faz 2** kapsamındaki **Conditional DDPM / DDIM Diffusion** (`rna-diffusion-clean` branch'i) modelinin Google Colab üzerinde eğitimi, DDIM tahmini, birleşik değerlendirmesi ve **Faz 1 (`rna-cgan-clean`) + `P(d)` Baz Çizgisi** ile **%100 adil kıyaslaması** için hazırlanmıştır.

### Adil Karşılaştırma Garantisi (Faz 1 ile Birebir Eşleşen Omurga):
- Aynı veri bölünmesi (`splits.txt`: train/val/test kromozomları)
- Aynı 1D + 2D `feature_pack` temsili (`representative` + `kmer7_revcomp_cosine`), `flankPad='all'`, `gridStride=16`, SAT aktif filtre
- Aynı `p99_log10` hedef normalizasyonu (`USE_LOGP_NORM=False` ile `--noLogpNorm` açılıp kapatılabilir) ve `raised_cosine_peak` yamalı birleştirme
- Aynı `condition_encoder` (`SplitChromatinStreamsLayer` + paylaşımlı Siamese `Conv1D` + `Pairwise2DEmbeddingLayer` -> `(W, W, 16)` `x_emb`) ve aynı `UNet` derinlik/filtre omurgası (`GroupNorm`, `CosineDecay(alpha=0.05)`)
- **Fark (yalnızca Faz 2 model mekanizması)**: Discriminator ve Adversarial Loss kaldırılmıştır; yerine 1000 adımlı lineer $\beta_t$ Diffusion + `SpatialTimeEncoding` + seçilebilir `--lossMode` (`current` / `paper` / `balanced`) ve 25 adımlı deterministik DDIM çıkarımı kullanılır.

In [ ]:
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/nehirvardar/Hi-cGAN.git'
REPO_DIR = '/content/Hi-cGAN'
BRANCH = 'rna-diffusion-clean'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q openpyxl cooler seaborn scikit-image tqdm
!pip install -q -e . --no-deps
!nvidia-smi || true

## 1. Veri & Deney Parametreleri (Faz 1 ile Eşitlenmiştir)

In [ ]:
from datetime import datetime
import glob

DRIVE_ROOT = '/content/drive/MyDrive/hicgan_runs'
DATA_DIR = f'{REPO_DIR}/hicgan/data/rna_gt'
CHROMATIN_DIR = f'{REPO_DIR}/data/feature_pack'
SPLIT_FILE = f'{REPO_DIR}/hicgan/data/splits.txt'

# Eğitim ve tahmin çıktıları Google Drive'a yazılır
RUN_TAG = datetime.now().strftime('diff_%Y%m%d_%H%M%S')
RUN_DIR = f'{DRIVE_ROOT}/runs_phase2_diffusion/{RUN_TAG}'
TRAIN_OUT = f'{RUN_DIR}/train_run'
PRED_OUT = f'{RUN_DIR}/predict_run'
os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs(PRED_OUT, exist_ok=True)

# Faz 1 ile birebir aynı veri & eğitim hiperparametreleri
BIN_SIZE = 250000
WINDOW_SIZE = 64
GRID_STRIDE = 16
FEATURE_SET = 'representative'
USE_PAIR_FEATURES = True
USE_LOGP_NORM = True  # True: p99_log10 (ln(1+10x)/P99), False: --noLogpNorm (ham sayımlar, normalizasyon yok)
TARGET_NORM = 'p99_log10' if USE_LOGP_NORM else 'none'
BLEND_MODE = 'raised_cosine_peak'
DROPOUT = 0.5  # Faz 1 Colab run'ı DROPOUT=0.5 ile eğitildiyse birebir regülarizasyon eşitliği için 0.5
EPOCHS = 50
BATCH_SIZE = 16
LR = 2e-5
EARLY_STOP = 10
SEED = 42

# Faz 2 Diffusion hiperparametreleri
# LOSS_MODE seçenekleri:
#   'current'  : mevcut (fg=15, peak=10, x0=1.0, bg_l1=0.5)
#   'paper'    : orijinal DDPM makale kaybı (ağırlıksız eps MSE: fg=0, peak=0, x0=0, bg_l1=0)
#   'balanced' : dengeli + arka plan cezalı (fg=3, peak=2, x0=0.5, bg_l1=1.0)
LOSS_MODE = 'current'
TIMESTEPS = 1000
SAMPLE_STEPS = 25

for p in (DATA_DIR, CHROMATIN_DIR, SPLIT_FILE):
    assert os.path.exists(p), f'Eksik yol: {p}'

print(f'RUN_DIR     : {RUN_DIR}')
print(f'DATA_DIR    : {DATA_DIR} (exists={os.path.exists(DATA_DIR)})')
print(f'CHROMATIN   : {CHROMATIN_DIR} (exists={os.path.exists(CHROMATIN_DIR)})')
print(f'SPLIT_FILE  : {SPLIT_FILE} (exists={os.path.exists(SPLIT_FILE)})')

## 2. Faz 2 Diffusion Modelini Eğitme (`hicgan.training`)

In [ ]:
pair_flag = '--usePairFeatures' if USE_PAIR_FEATURES else '--noPairFeatures'
logp_flag = '--logpNorm' if USE_LOGP_NORM else '--noLogpNorm'

!python -u -m hicgan.training \
    --dataDir "{DATA_DIR}" \
    --trainingChromatinFolders "{CHROMATIN_DIR}" \
    --validationChromatinFolders "{CHROMATIN_DIR}" \
    --splitFile "{SPLIT_FILE}" \
    --outputFolder "{TRAIN_OUT}" \
    --binSize {BIN_SIZE} \
    --windowSize {WINDOW_SIZE} \
    --windowMode grid \
    --gridStride {GRID_STRIDE} \
    --inputMode feature_pack \
    --featureSet "{FEATURE_SET}" \
    {pair_flag} \
    --targetNorm "{TARGET_NORM}" \
    {logp_flag} \
    --flankPadChromosomes all \
    --epochs {EPOCHS} \
    --batchSize {BATCH_SIZE} \
    --learningRate {LR} \
    --dropout {DROPOUT} \
    --earlyStoppingPatience {EARLY_STOP} \
    --lossMode "{LOSS_MODE}" \
    --timesteps {TIMESTEPS} \
    --sampleSteps {SAMPLE_STEPS} \
    --seed {SEED} \
    --keepTFRecords \
    --plotFrequency 10

## 3. Diffusion Eğitim Kayıp (Loss) Eğrileri Analizi

In [ ]:
from IPython.display import Image, display
from hicgan.lib import evaluation

df_loss = evaluation.parse_training_losses(TRAIN_OUT)
display(df_loss.tail(10))
loss_png = evaluation.plot_training_loss_analysis(
    df_loss,
    eval_out_dir=os.path.join(TRAIN_OUT, 'loss_analysis'),
    run_name=RUN_TAG
)
if loss_png and os.path.exists(loss_png):
    display(Image(filename=loss_png))

## 4. Test Kromozomları Üzerinde 25-Adımlı DDIM Tahmini ve Tekli Değerlendirme (`hicgan.predict --runEval`)

In [ ]:
MODEL_PATH = os.path.join(TRAIN_OUT, 'denoiser_best.keras')
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = os.path.join(TRAIN_OUT, 'denoiser_final.keras')
print('Kullanılan Denoiser:', MODEL_PATH)

pair_flag = '--usePairFeatures' if USE_PAIR_FEATURES else '--noPairFeatures'

!python -u -m hicgan.predict \
    --trainedModel "{MODEL_PATH}" \
    --dataDir "{DATA_DIR}" \
    --gtDir "{DATA_DIR}" \
    --predictionChromosomesFolders "{CHROMATIN_DIR}" \
    --splitFile "{SPLIT_FILE}" \
    --split test \
    --outputFolder "{PRED_OUT}" \
    --binSize {BIN_SIZE} \
    --windowSize {WINDOW_SIZE} \
    --windowMode grid \
    --gridStride {GRID_STRIDE} \
    --inputMode feature_pack \
    --featureSet "{FEATURE_SET}" \
    {pair_flag} \
    --flankPadChromosomes all \
    --batchSize {BATCH_SIZE} \
    --blendMode "{BLEND_MODE}" \
    --timesteps {TIMESTEPS} \
    --sampleSteps {SAMPLE_STEPS} \
    --mode all \
    --runEval

In [ ]:
import pandas as pd

eval_dir = os.path.join(PRED_OUT, 'evaluation')
sc_csv = os.path.join(eval_dir, 'core_metrics_scorecard.csv')
if os.path.exists(sc_csv):
    display(pd.read_csv(sc_csv))
for png_p in sorted(glob.glob(os.path.join(eval_dir, '*.png'))):
    display(Image(filename=png_p))

## 5. Faz 1 (`rna-cgan-clean`) vs Faz 2 (`rna-diffusion-clean`) Adil Kıyaslama

Bu bölüm **birebir aynı test kromozom kesişimi** üzerinde:
1. **Fairness Audit (`fairness_audit.json`)**: `binSize`, `windowSize`, `windowMode`, `gridStride`, `inputMode`, `featureSet`, `usePairFeatures`, `targetNorm`, `lossMode`, `logpNorm`, `flankPadChromosomes`, `blendMode` ve `dropout` eşitliğini doğrular.
2. **Liderlik Tablosu (`comparison_leaderboard.csv`)** ve **Kromozom Bazlı Δ Tablosu (`per_chromosome_comparison.csv`)** üretir (`Pearson r`, `Spearman ρ`, `Katmanlı r(d)/ρ(d)`, `MAE`, `MSE`, `RMSE`, `SSIM`, `PSNR`).
3. **Karşılaştırma Çubukları, Per-Chrom Pearson/Spearman, Katmanlı Mesafe Eğrileri ve 5 Panelli Isı Haritaları (`Girdi (Input) | GT | cGAN | Diffusion | Diff - cGAN`)** çizer.

In [ ]:
# Faz 1 cGAN run dizininizi buraya verin (veya Drive'da otomatik aramayı kullanın)
# Not: colab_rna_cgan_clean.ipynb varsayılan olarak /content/drive/MyDrive/hicgan_runs/rna_cgan_clean_* altına yazar.
CGAN_RUN_DIR = ''

if not CGAN_RUN_DIR:
    cand_roots = [
        '/content/drive/MyDrive/hicgan_runs',
        '/content/drive/MyDrive/Hi-cGAN/runs_phase1_cgan',
        '/content/drive/MyDrive/Hi-cGAN/output_models',
        '/content/drive/MyDrive/Hi-cGAN',
    ]
    cands = []
    for cr in cand_roots:
        if os.path.isdir(cr):
            for d in sorted(glob.glob(os.path.join(cr, '*'))):
                has_npz = (
                    os.path.isdir(os.path.join(d, 'test_eval', 'npz_matrices'))
                    or os.path.isdir(os.path.join(d, 'predict_run', 'npz_matrices'))
                    or os.path.isdir(os.path.join(d, 'npz_matrices'))
                )
                if has_npz and 'diff' not in os.path.basename(d).lower():
                    cands.append(d)
    if cands:
        CGAN_RUN_DIR = cands[-1]

print('CGAN_RUN_DIR :', CGAN_RUN_DIR)
print('DIFF_RUN_DIR :', RUN_DIR)

if CGAN_RUN_DIR and os.path.exists(CGAN_RUN_DIR):
    CMP_OUT = os.path.join(RUN_DIR, 'comparison_cgan_vs_diffusion')
    cmp_res = evaluation.compare_two_models(
        cgan_dir=CGAN_RUN_DIR,
        diffusion_dir=RUN_DIR,
        npz_data_dir=DATA_DIR,
        eval_out_dir=CMP_OUT,
        bin_size=BIN_SIZE,
        cgan_name='Phase 1: cGAN (rna-cgan-clean)',
        diffusion_name='Phase 2: Diffusion (rna-diffusion-clean)',
        chromatin_dir=CHROMATIN_DIR,
        feature_set=FEATURE_SET,
        use_pair_features=USE_PAIR_FEATURES,
    )
    print('Fairness Audit:', cmp_res['fairness_audit']['fair_comparison'])
    if cmp_res['fairness_audit']['warnings']:
        print('Fairness Warnings:', cmp_res['fairness_audit']['warnings'])
    display(cmp_res['leaderboard'])
    display(cmp_res['per_chromosome'])

    for img_key in ['radar_bar_png', 'per_chrom_png', 'distance_curves_png']:
        p = cmp_res['files'].get(img_key)
        if p and os.path.exists(p):
            display(Image(filename=p))
    for hm in cmp_res['files'].get('heatmap_pngs', []):
        if os.path.exists(hm):
            display(Image(filename=hm))
else:
    print('Faz 1 cGAN run klasörü bulunamadı; CGAN_RUN_DIR değişkenini manuel olarak tanımlayıp hücreyi yeniden çalıştırın.')